In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/Dataset.csv')

print("Dataset shape:",df.shape) # show number of rows and cols
print("\n Columns name",df.columns.tolist())
print("\nData types and non-null values:")
df.info() #shows data types and non-missing counts.
print("\nFirst five rows:")
display(df.head())

Dataset shape: (26305, 6)

 Columns name ['timestamp', 'cpu_utilization', 'memory_usage', 'storage_usage', 'workload', 'Resource Allocation']

Data types and non-null values:
<class 'pandas.DataFrame'>
RangeIndex: 26305 entries, 0 to 26304
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   timestamp            26305 non-null  str    
 1   cpu_utilization      23486 non-null  float64
 2   memory_usage         23234 non-null  float64
 3   storage_usage        22999 non-null  float64
 4   workload             26305 non-null  float64
 5   Resource Allocation  23733 non-null  float64
dtypes: float64(5), str(1)
memory usage: 1.7 MB

First five rows:


,timestamp,cpu_utilization,memory_usage,storage_usage,workload,Resource Allocation
0,2022-01-01 00:00:00,39.963210,37.120463,77.268825,87.597635,51.911114
1,2022-01-01 01:00:00,86.060728,51.540619,56.655229,100.000000,61.732726
2,2022-01-01 02:00:00,68.566681,NaN,62.418368,100.000000,75.685323
3,2022-01-01 03:00:00,57.903428,51.213742,95.346718,100.000000,67.873120
4,2022-01-01 04:00:00,NaN,74.822334,84.865431,93.433681,65.148501


In [2]:
#check missing values and duplication

print("Missing values in each column:")
print(df.isnull().sum()) #count NaN values in cols

print("\nTotal duplicated rows:")
print(df.duplicated().sum()) #count rows that have exact same values

Missing values in each column:
timestamp                 0
cpu_utilization        2819
memory_usage           3071
storage_usage          3306
workload                  0
Resource Allocation    2572
dtype: int64

Total duplicated rows:
0


In [3]:
#we will remove rows where target (cpu) is missing
df_model=df.dropna(subset=["cpu_utilization"]).copy()

#then define X and y 

X=df_model[["memory_usage","storage_usage","workload"]]
y=df_model[["cpu_utilization"]]

#printing number of rows before and after 
print("Rows before:",len(df))
print("\nRows after :", len(df_model))

#missing values should be null in y

print("\nMissing target values:")
print(y.isnull().sum())



Rows before: 26305

Rows after : 23486

Missing target values:
cpu_utilization    0
dtype: int64


In [4]:
# Calculate the splitting point (80% training, 20% testing)
split_index = int(len(X) * 0.8)

# Split X (input features)
X_train = X.iloc[:split_index].copy()
X_test = X.iloc[split_index:].copy()

# Split y (target values)
y_train = y.iloc[:split_index].copy()
y_test = y.iloc[split_index:].copy()

# Check the number of rows
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("Training missing values:")
print(X_train.isnull().sum())

print("\nTesting missing values:")
print(X_test.isnull().sum())

Training rows: 18788
Testing rows: 4698
Training missing values:
memory_usage     2200
storage_usage    2351
workload            0
dtype: int64

Testing missing values:
memory_usage     540
storage_usage    575
workload           0
dtype: int64


In [5]:
from sklearn.impute import SimpleImputer

# Use the median because it is less affected by extreme values than the mean
imputer = SimpleImputer(strategy="median")

# Calculate medians using only the training data and fill missing values
X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

# Use the same medians to fill missing values in the testing data
X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# Check that no missing values remain
print("Training missing values:")
print(X_train.isnull().sum())

print("\nTesting missing values:")
print(X_test.isnull().sum())

Training missing values:
memory_usage     0
storage_usage    0
workload         0
dtype: int64

Testing missing values:
memory_usage     0
storage_usage    0
workload         0
dtype: int64


In [6]:
import torch

# Convert training and testing features into PyTorch tensors
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)

# Convert training and testing targets into PyTorch tensors
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

# Check tensor shapes
print("X_train shape:", X_train_tensor.shape)
print("X_test shape:", X_test_tensor.shape)
print("y_train shape:", y_train_tensor.shape)
print("y_test shape:", y_test_tensor.shape)

X_train shape: torch.Size([18788, 3])
X_test shape: torch.Size([4698, 3])
y_train shape: torch.Size([18788, 1])
y_test shape: torch.Size([4698, 1])


In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Create datasets and load them in batches
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

trainloader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True 
)

testloader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)